In [1]:
import geopandas as gpd

beijing = gpd.read_file('Beijing/北京市全部_Polygon.shp')
beijing

,NAME,geometry
0,东城区,"POLYGON ((116.38142 39.95952, 116.38323 39.959..."
1,西城区,"POLYGON ((116.37436 39.86975, 116.36765 39.869..."
2,朝阳区,"POLYGON ((116.43943 39.87721, 116.43946 39.877..."
3,朝阳区,"POLYGON ((116.59788 40.05190, 116.59463 40.051..."
4,丰台区,"POLYGON ((116.16084 39.88744, 116.16269 39.885..."
5,石景山区,"POLYGON ((116.25306 39.89545, 116.25265 39.895..."
6,海淀区,"POLYGON ((116.31968 39.89549, 116.31926 39.895..."
7,门头沟区,"POLYGON ((115.50341 40.06429, 115.50625 40.065..."
8,房山区,"POLYGON ((116.21516 39.57774, 116.21466 39.577..."
9,通州区,"POLYGON ((116.71950 39.62289, 116.71915 39.625..."


In [2]:
grid = gpd.read_file('grid/grid.shp')
grid

,行,列,环,区,街道,标识符,pop,road,poi,ershoufang,zujin,4,8,11,geometry
0,54,2,6,房山,阎村镇,1,1914.141414,24.873974,94.0,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((422959.138 4393556.876, 422959.138 4..."
1,54,3,6,房山,阎村镇,2,439.000000,2.024137,8.0,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((423959.138 4393556.876, 423959.138 4..."
2,54,4,6,房山,良乡地区办事处,3,560.000000,2.583739,2.0,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((424959.138 4393556.876, 424959.138 4..."
3,54,5,6,房山,良乡地区办事处,4,1885.000000,2.659241,52.0,0.0,31.250000,0.000000,31.250000,0.000000,"POLYGON ((425959.138 4393556.876, 425959.138 4..."
4,54,6,6,房山,良乡地区办事处,5,1036.633663,1.749587,34.0,10000.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((426959.138 4393556.876, 426959.138 4..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2369,1,18,6,昌平,百善镇,2370,440.000000,3.796777,6.0,0.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((438959.138 4446556.876, 438959.138 4..."
2370,1,19,6,昌平,百善镇,2371,4319.191919,0.721418,7.0,33418.0,0.000000,0.000000,0.000000,0.000000,"POLYGON ((439959.138 4446556.876, 439959.138 4..."
2371,1,20,6,昌平,百善镇,2372,4698.019802,3.347467,59.0,29284.0,71.423934,71.423934,0.000000,0.000000,"POLYGON ((440959.138 4446556.876, 440959.138 4..."
2372,1,21,6,昌平,百善镇,2373,331.683168,4.667819,3.0,0.0,56.967612,0.000000,54.595086,59.340138,"POLYGON ((441959.138 4446556.876, 441959.138 4..."


In [13]:
grid = grid.to_crs(3857)
beijing = beijing.to_crs(3857)

In [32]:

import numpy as np
max_r = int(grid["行"].max())
max_c = int(grid["列"].max())

sindex = grid.sindex  # R-tree

for _, row in beijing.iterrows():
    geom = row["geometry"]
    name = str(row["NAME"])

    # Quickly pre-select potentially intersecting cells via bounding box …
    subset = grid[grid.intersects(geom)]  # exact test
    
    tensor = np.zeros((max_r + 1, max_c + 1), dtype=np.uint8)
    tensor[subset["行"].to_numpy(), subset["列"].to_numpy()] = 1

    print(subset["行"].to_numpy())
    print(subset["列"].to_numpy())
    print(tensor.sum())
    if name[:-1] in ['密云','平谷','延庆','怀柔','石景山','门头沟', '西城', '东城']:
        continue
    path = name + "/mask.txt"
    delimiter = ","
    np.savetxt(path, tensor, fmt="%d", delimiter=delimiter)
    
    # print(f"✓ wrote {path}  ({tensor.sum():,d} intersecting cells)")
    # break
    

[36 36 36 35 35 35 35 35 34 34 34 34 34 34 33 33 33 33 33 32 32 32 32 32
 32 31 31 31 31 31 31 30 30 30 30 30 29 29 29 29 29 28 28 28 28 28 27 27
 27 27 27 27 26 26 26 26 26 26 25 25 25 25 24 24 24 24 23 23]
[26 27 28 25 26 27 28 30 26 27 28 29 30 31 27 28 29 30 31 26 27 28 29 30
 31 26 27 28 29 30 31 26 27 28 29 30 26 27 28 29 30 26 27 28 29 30 26 27
 28 29 30 31 26 27 28 29 30 31 26 27 28 29 26 27 28 29 27 28]
68
[35 35 35 35 35 34 34 34 34 34 34 34 34 33 33 33 33 33 33 33 33 32 32 32
 32 32 32 32 32 31 31 31 31 31 31 30 30 30 30 30 30 29 29 29 29 29 29 29
 28 28 28 28 28 28 28 27 27 27 27 27 27 26 26 26 26 26 26 25 25 25 24 24
 24 23 23]
[22 23 24 25 26 20 21 22 23 24 25 26 27 20 21 22 23 24 25 26 27 20 21 22
 23 24 25 26 27 21 22 23 24 25 26 21 22 23 24 25 26 21 22 23 24 25 26 27
 21 22 23 24 25 26 27 21 22 23 24 25 26 21 22 23 24 25 26 24 25 26 24 25
 26 25 26]
75
[41 41 41 41 41 40 40 40 40 40 40 40 40 40 40 40 40 39 39 39 39 39 39 39
 39 39 39 39 39 39 39 39 39 38 38 38 38 38 38

In [28]:
sum1 = np.zeros((55, 54))
for district in ['密云','平谷','延庆','怀柔','石景山','门头沟', '西城', '东城']:
    ts = np.loadtxt(district+'区.txt', delimiter=',', dtype=np.uint8)
    sum1 += ts
sum1.sum()

321.0

In [33]:
np.savetxt('剩余五区/mask.txt', sum1, fmt="%d", delimiter=delimiter)

In [24]:
import torch

ts = torch.tensor(ts)
ts.shape

torch.Size([55, 54])